In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


# Kvasir-VQA x1 — Retrieval-augmented BLIP-2 (evaluation)

Lightweight RAG-style evaluation: retrieve similar samples by question text, format a context prompt, and run BLIP-2 (zero/few-shot) without fine-tuning. This is an evaluation notebook, not training.

In [2]:

from pathlib import Path
import json
import random
from typing import List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from PIL import Image

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


2026-01-30 01:42:29.979435: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-30 01:42:29.979470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-30 01:42:29.980436: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-30 01:42:29.986710: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 01:42:30.804010: W tensorflow/compiler/tf2

In [3]:

# Paths & config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "09_rag_blip2_eval" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip2-opt-2.7b"  # swap to lighter if needed
MAX_GEN_TOKENS = 16
RETRIEVE_K = 3
TOP_K_ANSWERS = 20  # restrict retrieval index to common answers to keep context clean
MAX_EVAL_SAMPLES = 200  # limit test set for quick eval; set None for full

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Out dir: /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/09_rag_blip2_eval/out
Device: cuda


In [4]:

# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta = meta.dropna(subset=["question", "answer", "image_path"]).reset_index(drop=True)
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

answer_counts = meta["answer"].value_counts()
top_answers = set(answer_counts.head(TOP_K_ANSWERS).index)
index_df = meta[meta["answer"].isin(top_answers)].reset_index(drop=True)

# Build splits
train_df = index_df[index_df["split"] == "train"].reset_index(drop=True)
val_df   = index_df[index_df["split"] == "validation"].reset_index(drop=True)
test_df  = index_df[index_df["split"] == "test"].reset_index(drop=True)

if MAX_EVAL_SAMPLES:
    test_df = test_df.sample(min(MAX_EVAL_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 14370, 'val': 0, 'test': 200}


In [5]:

# Build retriever on questions (TF-IDF + cosine NN)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
train_qs = train_df["question"].tolist()
vec_train = vectorizer.fit_transform(train_qs)

nn = NearestNeighbors(n_neighbors=RETRIEVE_K, metric="cosine")
nn.fit(vec_train)

print("Retriever fitted on", len(train_qs), "questions")


Retriever fitted on 14370 questions


In [6]:

# Load BLIP-2
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
model.config.use_cache = False


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 28.88 MiB is free. Process 455179 has 604.00 MiB memory in use. Process 464211 has 404.00 MiB memory in use. Process 479757 has 480.00 MiB memory in use. Process 492950 has 8.04 GiB memory in use. Process 575273 has 2.39 GiB memory in use. Process 603461 has 1.04 GiB memory in use. Including non-PyTorch memory, this process has 1.81 GiB memory in use. Of the allocated memory 1.54 GiB is allocated by PyTorch, and 55.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Retrieval helper
def retrieve_context(question: str) -> List[str]:
    vec = vectorizer.transform([question])
    dist, idx = nn.kneighbors(vec, n_neighbors=RETRIEVE_K)
    rows = train_df.iloc[idx[0]]
    lines = [f"Q: {r['question']} A: {r['answer']}" for _, r in rows.iterrows()]
    return lines

PROMPT_TEMPLATE = "You are a medical VQA assistant. Use the retrieved examples and the image to answer the question concisely.\nExamples:\n{examples}\nQuestion: {question}\nAnswer:"

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    ctx = "\n".join(retrieve_context(row["question"]))
    prompt = PROMPT_TEMPLATE.format(examples=ctx, question=row["question"])
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

In [ ]:

# Evaluate on test subset
preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="RAG BLIP2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

results = {"pred": preds, "ref": refs}
pd.DataFrame(results).to_csv(OUT_DIR / "predictions.csv", index=False)

# Simple metrics: exact match rate on top-K answers
exact = [p.lower() == r.lower() for p, r in zip(preds, refs)]
acc = sum(exact) / len(exact) if exact else 0

metrics = {"exact_match": acc, "n": len(test_df)}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)
